# Read Data from silver container

In [0]:
df = spark.read.format("delta").load("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")
display(df)

## Create a temporary view to perform sql operations

In [0]:
df.createOrReplaceTempView("sales")

In [0]:
df_src = spark.sql("""
          select distinct Model_ID, ModelType, Product_Name
          from sales
          """)
display(df_src)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import monotonically_increasing_id, cast

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_model"):
    df_sink = spark.sql("""
                        select dim_model_key, Model_ID, ModelType, Product_Name
                        from carsalescatalog.gold.dim_model
                        """)
    
else:
    df_sink = spark.sql("""
          select 1 as dim_model_key, Model_ID, ModelType, Product_Name
          from sales
          where 1=0
          """)
    
    

In [0]:
df_sink.display()

In [0]:
df_new_data = df_src.join(df_sink, df_sink["Model_ID"] == df_src["Model_ID"], "left")\
    .select(df_src["Model_ID"], df_src["ModelType"], df_src["Product_Name"], df_sink["dim_model_key"])

df_new_data.display()

In [0]:
df_new_records = df_new_data.filter(df_new_data.dim_model_key.isNull())
df_old_records = df_new_data.filter(df_new_data.dim_model_key.isNotNull())

# Add surrogate dim key for new records

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_model"):
    max_value = spark.sql("""
                        select max(dim_model_key)
                        from carsalescatalog.gold.dim_model
                        """).collect()[0][0]
    
else:
    max_value = 0

In [0]:
df_new_records = df_new_records.withColumn("dim_model_key", max_value + monotonically_increasing_id() + 1)
display(df_new_records)

# Appending new and old records after adding surrogate key

In [0]:
df_final = df_new_records.unionByName(df_old_records)
display(df_final)

# Update model dimension table

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_model/"):
    deltatable = DeltaTable.forPath(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_model/")

    deltatable.alias("trg").merge(df_final.alias("src"), "trg.Model_ID = src.Model_ID")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format("delta").mode("overwrite").save("abfss://gold@adlscarsales.dfs.core.windows.net/dim_model/")

    spark.sql("""
              create table carsalescatalog.gold.dim_model
              using delta
              location 'abfss://gold@adlscarsales.dfs.core.windows.net/dim_model/'
              """)